# Vietnamese Handwriting OCR - Kaggle OCR + Token Corrector




In [1]:
import os
import sys
import re
import io
import csv
import math
import time
import json
import shutil
import zipfile
import hashlib
import random
import string
import warnings
import unicodedata
from collections import Counter, defaultdict
import pkgutil
import subprocess
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

def ensure_package(package_name: str, import_name: str | None = None) -> None:
    import_name = import_name or package_name
    if pkgutil.find_loader(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

ensure_package("editdistance")
ensure_package("scikit-learn", "sklearn")
ensure_package("rapidfuzz")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import editdistance
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import backend as K
from tensorflow.keras import mixed_precision
from IPython.display import display
from rapidfuzz import process as rf_process, fuzz as rf_fuzz

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

try:
    tf.config.optimizer.set_experimental_options({"layout_optimizer": False})
except Exception as exc:
    print("layout_optimizer config skipped:", exc)

if gpus:
    mixed_precision.set_global_policy("mixed_float16")

strategy = tf.distribute.MirroredStrategy() if len(gpus) > 1 else tf.distribute.get_strategy()

print("TensorFlow:", tf.__version__)
print("GPUs      :", len(gpus))
print("Strategy  :", type(strategy).__name__)
print("Policy    :", mixed_precision.global_policy())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.4 MB/s eta 0:00:00


2026-03-22 09:24:57.152847: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774171497.641729      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774171497.787205      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774171498.828829      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774171498.828922      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774171498.828924      24 computation_placer.cc:177] computation placer alr

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
TensorFlow: 2.19.0
GPUs      : 2
Strategy  : MirroredStrategy
Policy    : <DTypePolicy "mixed_float16">


I0000 00:00:1774171534.126417      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774171534.129257      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [2]:
CFG = {
    "run_mode": "ocr_plus_token_corrector",
    "target_height": 96,
    "downsample_factor": 4,
    "per_device_batch_size": 2,
    "initial_lr": 3e-5,
    "val_size": 0.12,
    "max_width": 2560,
    "min_width": 64,
    "width_pad_multiple": 16,
    "fixed_eval_width": 2304,
    "preprocess_profiles": ["legacy_clean"],
    "dataset_root": None,
    "weights_path": "/kaggle/input/models/trinhminhkhoak18hcm/ocr-train-chuan-checkpoint-weights/gguf/default/1/ocr-train-chuan_checkpoint.weights.h5",
    "train10_artifact": None,
    "resume_artifact": None,
    "camera_hard_eval_root": None,
    "camera_hard_sample_size": 256,
    "camera_hard_enabled": True,
    "preview_samples": 4,
    "token_min_freq": 2,
    "token_corrector_epochs": 20,
    "token_corrector_batch_size": 256,
    "token_corrector_embedding_dim": 96,
    "token_corrector_hidden_dim": 192,
    "token_corrector_train_repeats": 4,
    "token_corrector_val_ratio": 0.08,
    "token_corrector_max_len": 28,
    "token_corrector_accept_margin": 0.18,
    "token_corrector_min_avg_prob": 0.82,
    "token_similarity_threshold": 72.0,
    "token_same_lexicon_guard": True,
    "max_token_corrections_per_line": 1,
    "output_root": "/kaggle/working/ocr_token_corrector_run",
}

CFG["global_batch_size"] = CFG["per_device_batch_size"] * strategy.num_replicas_in_sync
OUTPUT_ROOT = Path(CFG["output_root"])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

PATHS = {
    "metadata_cache": OUTPUT_ROOT / "ocr_metadata_cache.csv",
    "config_json": OUTPUT_ROOT / "run_config.json",
    "selection_json": OUTPUT_ROOT / "selected_sentence_pipeline.json",
    "compare_csv": OUTPUT_ROOT / "sentence_eval_compare.csv",
    "token_history_csv": OUTPUT_ROOT / "token_corrector_history.csv",
    "token_assets_json": OUTPUT_ROOT / "token_corrector_assets.json",
    "diagnostics_json": OUTPUT_ROOT / "stage_diagnostics.json",
    "token_corrector_model": OUTPUT_ROOT / "token_corrector.keras",
    "export_model": OUTPUT_ROOT / "ocr_sentence_export.keras",
    "selected_weights": OUTPUT_ROOT / "selected_sentence.weights.h5",
    "bundle_zip": OUTPUT_ROOT / "artifact_bundle.zip",
    "metrics_dir": OUTPUT_ROOT / "metrics",
    "predictions_dir": OUTPUT_ROOT / "predictions",
}

PATHS["metrics_dir"].mkdir(parents=True, exist_ok=True)
PATHS["predictions_dir"].mkdir(parents=True, exist_ok=True)

with open(PATHS["config_json"], "w", encoding="utf-8") as f:
    json.dump(CFG, f, ensure_ascii=False, indent=2)

print(json.dumps(CFG, ensure_ascii=False, indent=2))


{
  "run_mode": "ocr_plus_token_corrector",
  "target_height": 96,
  "downsample_factor": 4,
  "per_device_batch_size": 2,
  "initial_lr": 3e-05,
  "val_size": 0.12,
  "max_width": 2560,
  "min_width": 64,
  "width_pad_multiple": 16,
  "fixed_eval_width": 2304,
  "preprocess_profiles": [
    "legacy_clean"
  ],
  "dataset_root": null,
  "weights_path": "/kaggle/input/models/trinhminhkhoak18hcm/ocr-train-chuan-checkpoint-weights/gguf/default/1/ocr-train-chuan_checkpoint.weights.h5",
  "train10_artifact": null,
  "resume_artifact": null,
  "camera_hard_eval_root": null,
  "camera_hard_sample_size": 256,
  "camera_hard_enabled": true,
  "preview_samples": 4,
  "token_min_freq": 2,
  "token_corrector_epochs": 20,
  "token_corrector_batch_size": 256,
  "token_corrector_embedding_dim": 96,
  "token_corrector_hidden_dim": 192,
  "token_corrector_train_repeats": 4,
  "token_corrector_val_ratio": 0.08,
  "token_corrector_max_len": 28,
  "token_corrector_accept_margin": 0.18,
  "token_corrector_

In [3]:
SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]

def existing_search_roots():
    return [root for root in SEARCH_ROOTS if root.exists()]

def maybe_unpack_resume_bundle(manual_path=None):
    if manual_path:
        bundle = Path(manual_path)
        if bundle.is_file() and bundle.suffix.lower() == ".zip":
            target = OUTPUT_ROOT / "resume_bundle_input"
            if target.exists():
                shutil.rmtree(target)
            target.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(bundle, "r") as zf:
                zf.extractall(target)
            return target
        if bundle.is_dir():
            return bundle

    for root in existing_search_roots():
        for bundle in root.rglob("resume_bundle.zip"):
            target = OUTPUT_ROOT / "resume_bundle_input"
            if target.exists():
                shutil.rmtree(target)
            target.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(bundle, "r") as zf:
                zf.extractall(target)
            return target

    # Support the case where the user uploaded the extracted resume folder as a dataset.
    for root in existing_search_roots():
        for balanced_path in root.rglob("best_balanced.weights.h5"):
            parent = balanced_path.parent
            if (parent / "best.weights.h5").exists():
                return parent
    return None

def discover_dataset_root(manual_path=None):
    if manual_path:
        root = Path(manual_path)
        if (root / "labels.json").exists() and (root / "data").exists():
            return root
        raise FileNotFoundError(f"Dataset root khong hop le: {root}")

    candidates = []
    for root in existing_search_roots():
        for labels_path in root.rglob("labels.json"):
            dataset_root = labels_path.parent
            if (dataset_root / "data").exists():
                score = 0
                lower = str(dataset_root).lower()
                if "vn_handwritten_images" in lower:
                    score += 5
                if "hand" in lower or "written" in lower:
                    score += 2
                candidates.append((score, str(dataset_root), dataset_root))
    if not candidates:
        raise FileNotFoundError("Khong tim thay dataset co labels.json va data/")
    candidates.sort(key=lambda x: (-x[0], x[1]))
    return candidates[0][2]

def discover_weights(manual_path=None):
    if manual_path:
        p = Path(manual_path)
        if p.exists():
            return p
        raise FileNotFoundError(f"Khong tim thay weights: {p}")

    preferred = {
        "ocr-train-chuan_checkpoint.weights.h5": 8,
        "best.weights.h5": 6,
        "latest.weights.h5": 5,
    }
    candidates = []
    for root in existing_search_roots():
        for path in root.rglob("*.h5"):
            if not path.is_file():
                continue
            score = preferred.get(path.name, 0)
            low = path.name.lower()
            if "checkpoint" in low:
                score += 2
            if "best" in low:
                score += 2
            if "ocr" in low:
                score += 1
            candidates.append((score, str(path), path))
    if not candidates:
        return None
    candidates.sort(key=lambda x: (-x[0], x[1]))
    return candidates[0][2]

def discover_resume_dir(preferred_root=None):
    search_roots = []
    if preferred_root and Path(preferred_root).exists():
        search_roots.append(Path(preferred_root))
    search_roots.extend(existing_search_roots())

    candidates = []
    for root in search_roots:
        for ckpt_marker in root.rglob("checkpoint"):
            ckpt_dir = ckpt_marker.parent
            if list(ckpt_dir.glob("ckpt-*.index")):
                score = 0
                if "resume" in str(ckpt_dir).lower():
                    score += 3
                if "ocr" in str(ckpt_dir).lower():
                    score += 1
                candidates.append((score, str(ckpt_dir), ckpt_dir))
    if not candidates:
        return None
    candidates.sort(key=lambda x: (-x[0], x[1]))
    return candidates[0][2]

def discover_camera_hard_root(manual_path=None):
    if manual_path:
        root = Path(manual_path)
        if (root / "labels.json").exists() and (root / "data").exists():
            return root
        raise FileNotFoundError(f"camera_hard_eval_root khong hop le: {root}")
    return None

RESUME_INPUT_ROOT = maybe_unpack_resume_bundle(CFG["resume_artifact"])
DATASET_ROOT = discover_dataset_root(CFG["dataset_root"])
LABELS_PATH = DATASET_ROOT / "labels.json"
IMAGE_DIR = DATASET_ROOT / "data"
WEIGHTS_PATH = discover_weights(CFG["weights_path"])
RESUME_DIR = discover_resume_dir(RESUME_INPUT_ROOT)
CAMERA_HARD_ROOT = discover_camera_hard_root(CFG["camera_hard_eval_root"])

print("DATASET_ROOT     =", DATASET_ROOT)
print("WEIGHTS_PATH     =", WEIGHTS_PATH)
print("RESUME_INPUT_ROOT=", RESUME_INPUT_ROOT)
print("RESUME_DIR       =", RESUME_DIR)
print("CAMERA_HARD_ROOT =", CAMERA_HARD_ROOT)


DATASET_ROOT     = /kaggle/input/datasets/huyyuh/3500-w/vn_handwritten_images
WEIGHTS_PATH     = /kaggle/input/models/trinhminhkhoak18hcm/ocr-train-chuan-checkpoint-weights/gguf/default/1/ocr-train-chuan_checkpoint.weights.h5
RESUME_INPUT_ROOT= None
RESUME_DIR       = None
CAMERA_HARD_ROOT = None


In [4]:
TRAIN10_SELECTION_PATH = None
TRAIN10_SELECTION = {}
DEFAULT_BACKBONE_WIDTH = 2304
DEFAULT_BACKBONE_DECODER = "greedy"
DEFAULT_BACKBONE_BEAM = 10
print("DEFAULT_BACKBONE_WIDTH =", DEFAULT_BACKBONE_WIDTH)
print("DEFAULT_BACKBONE_DECODER =", DEFAULT_BACKBONE_DECODER, DEFAULT_BACKBONE_BEAM)


DEFAULT_BACKBONE_WIDTH = 2304
DEFAULT_BACKBONE_DECODER = greedy 10


In [5]:
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    raw_labels = json.load(f)

def build_legacy_charset():
    vietnamese_vowels = [
        "a","à","á","ả","ã","ạ","ă","ằ","ắ","ẳ","ẵ","ặ","â","ầ","ấ","ẩ","ẫ","ậ",
        "e","è","é","ẻ","ẽ","ẹ","ê","ề","ế","ể","ễ","ệ",
        "i","ì","í","ỉ","ĩ","ị",
        "o","ò","ó","ỏ","õ","ọ","ô","ồ","ố","ổ","ỗ","ộ","ơ","ờ","ớ","ở","ỡ","ợ",
        "u","ù","ú","ủ","ũ","ụ","ư","ừ","ứ","ử","ữ","ự",
        "y","ỳ","ý","ỷ","ỹ","ỵ"
    ]
    vietnamese_consonants = ["b","c","d","đ","g","h","k","l","m","n","p","q","r","s","t","v","x"]
    upper_vowels = [c.upper() for c in vietnamese_vowels]
    upper_consonants = [c.upper() for c in vietnamese_consonants]
    digits = list("0123456789")
    punctuation = [" ", ".", ",", ":", ";", "!", "?", "-", "_", "(", ")", "[", "]", "/", "%", "&", "*", "=", "@", "$", "'", "\""]
    extra_chars = ["f", "j", "w", "z", "F", "J", "W", "Z", "#", "+"]
    return sorted(set(vietnamese_vowels + vietnamese_consonants + upper_vowels + upper_consonants + digits + punctuation + extra_chars))

char_list = build_legacy_charset()
char_to_idx = {c: i for i, c in enumerate(char_list)}
idx_to_char = {i: c for i, c in enumerate(char_list)}

dataset_chars = sorted(set("".join(raw_labels.values())))
missing_chars = sorted(set(dataset_chars) - set(char_list))
if missing_chars:
    raise ValueError(f"Dataset co ky tu ngoai legacy charset: {missing_chars[:20]}")

def encode_text(text: str):
    return [char_to_idx[c] for c in text if c in char_to_idx]

def build_source_group(filename: str) -> str:
    stem = Path(filename).stem
    if stem.endswith("_samples"):
        return f"samples:{stem.replace('_samples', '')}"
    if "_" in stem:
        first = stem.split("_")[0]
        if first.isdigit():
            return f"crop:{first}"
    if stem.isdigit():
        return f"single:{stem}"
    return f"misc:{stem}"

cache_path = PATHS["metadata_cache"]
use_cache = False
if cache_path.exists():
    cached = pd.read_csv(cache_path)
    if not cached.empty and "dataset_root" in cached.columns and str(cached["dataset_root"].iloc[0]) == str(DATASET_ROOT):
        records_df = cached.copy()
        use_cache = True

if not use_cache:
    rows = []
    for filename, text in raw_labels.items():
        image_path = IMAGE_DIR / filename
        if not image_path.exists():
            continue
        try:
            with Image.open(image_path) as img:
                width, height = img.size
        except Exception:
            continue
        scaled_width = int(round(width * CFG["target_height"] / max(1, height)))
        scaled_width = max(CFG["min_width"], min(CFG["max_width"], scaled_width))
        rows.append(
            {
                "dataset_root": str(DATASET_ROOT),
                "filename": filename,
                "path": str(image_path),
                "text": text,
                "label_length": len(text),
                "orig_width": width,
                "orig_height": height,
                "scaled_width": scaled_width,
                "source_group": build_source_group(filename),
            }
        )
    records_df = pd.DataFrame(rows)
    records_df.to_csv(cache_path, index=False, encoding="utf-8-sig")

records_df["required_width"] = (records_df["label_length"] + 2) * CFG["downsample_factor"]
records_df["encoded"] = records_df["text"].apply(encode_text)
records_df["encoded_length"] = records_df["encoded"].apply(len)
records_df["final_width_hint"] = records_df[["scaled_width", "required_width"]].max(axis=1)
records_df["length_bin"] = pd.cut(
    records_df["label_length"],
    bins=[-1, 10, 25, 50, 75, 100, 1000],
    labels=["<=10", "11-25", "26-50", "51-75", "76-100", ">100"],
)

splitter = GroupShuffleSplit(n_splits=1, test_size=CFG["val_size"], random_state=SEED)
train_idx, val_idx = next(splitter.split(records_df, groups=records_df["source_group"]))
train_df = records_df.iloc[train_idx].reset_index(drop=True)
val_df = records_df.iloc[val_idx].reset_index(drop=True)

print("All samples :", len(records_df))
print("Train samples:", len(train_df))
print("Val samples  :", len(val_df))
print("Unique train groups:", train_df["source_group"].nunique())
print("Unique val groups  :", val_df["source_group"].nunique())

group_overlap = set(train_df["source_group"]) & set(val_df["source_group"])
print("Shared groups:", len(group_overlap))

display(records_df.head(3))
display(pd.concat(
    [
        train_df["length_bin"].value_counts(normalize=True).rename("train_ratio"),
        val_df["length_bin"].value_counts(normalize=True).rename("val_ratio"),
    ],
    axis=1,
).fillna(0.0))


All samples : 15267
Train samples: 13944
Val samples  : 1323
Unique train groups: 2231
Unique val groups  : 305
Shared groups: 0


,dataset_root,filename,path,text,label_length,orig_width,orig_height,scaled_width,source_group,required_width,encoded,encoded_length,final_width_hint,length_bin
0,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,1.jpg,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,"Số 3 Nguyễn Ngọc Vũ, Hà Nội",27,1684,245,660,single:1,116,"[49, 179, 0, 19, 0, 44, 66, 80, 84, 167, 73, 0...",27,660,26-50
1,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,2.jpg,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,"Số 30 Nguyên Hồng, Láng Hạ, Đống Đa, Hà Nội",43,2030,276,706,single:2,180,"[49, 179, 0, 19, 16, 0, 44, 66, 80, 84, 108, 7...",43,706,26-50
2,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,3.jpg,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,"58 Thái Thịnh, Đống Đa, Hà Nội",30,2544,376,650,single:3,128,"[21, 24, 0, 50, 67, 103, 68, 0, 50, 67, 173, 7...",30,650,26-50


,train_ratio,val_ratio
length_bin,,
51-75,0.358003,0.363568
<=10,0.233362,0.288738
26-50,0.203385,0.201814
76-100,0.164730,0.117158
11-25,0.036862,0.027967
>100,0.003657,0.000756


In [6]:
def stable_int_seed(text: str) -> int:
    return int(hashlib.md5(text.encode("utf-8")).hexdigest()[:8], 16)

def align_width(width: int, multiple: int) -> int:
    return int(math.ceil(width / multiple) * multiple)

def add_shadow(gray: np.ndarray, rng: np.random.Generator, hard: bool) -> np.ndarray:
    h, w = gray.shape
    x = np.linspace(0, 1, w, dtype=np.float32)
    y = np.linspace(0, 1, h, dtype=np.float32)
    xv, yv = np.meshgrid(x, y)
    mask = xv if rng.random() < 0.5 else yv
    if rng.random() < 0.5:
        mask = 1.0 - mask
    strength = rng.uniform(0.20, 0.65) if hard else rng.uniform(0.35, 0.75)
    shadow = strength + (1.0 - strength) * mask
    out = gray.astype(np.float32) * shadow
    return np.clip(out, 0, 255).astype(np.uint8)

def add_glare(gray: np.ndarray, rng: np.random.Generator, hard: bool) -> np.ndarray:
    h, w = gray.shape
    canvas = gray.astype(np.float32).copy()
    for _ in range(1 if not hard else 2):
        center = (int(rng.integers(0, w)), int(rng.integers(0, h)))
        radius = int(rng.integers(max(20, w // 10), max(30, w // 4)))
        overlay = np.zeros_like(canvas)
        cv2.circle(overlay, center, radius, 255, -1)
        alpha = rng.uniform(0.08, 0.18) if not hard else rng.uniform(0.15, 0.28)
        canvas = cv2.addWeighted(canvas, 1.0, overlay, alpha, 0.0)
    return np.clip(canvas, 0, 255).astype(np.uint8)

def add_stains(gray: np.ndarray, rng: np.random.Generator, hard: bool) -> np.ndarray:
    canvas = gray.astype(np.float32).copy()
    h, w = gray.shape
    num_blobs = int(rng.integers(3, 8 if not hard else 12))
    for _ in range(num_blobs):
        center = (int(rng.integers(0, w)), int(rng.integers(0, h)))
        axes = (
            int(rng.integers(max(8, w // 40), max(12, w // (8 if hard else 10)))),
            int(rng.integers(max(6, h // 8), max(8, h // (2 if hard else 3)))),
        )
        color = float(rng.integers(150 if hard else 170, 255))
        angle = float(rng.integers(0, 180))
        cv2.ellipse(canvas, center, axes, angle, 0, 360, color, -1)
    alpha = 0.18 if hard else 0.12
    mixed = cv2.addWeighted(gray.astype(np.float32), 1.0 - alpha, canvas, alpha, 0)
    return np.clip(mixed, 0, 255).astype(np.uint8)

def add_occlusion(gray: np.ndarray, rng: np.random.Generator, hard: bool) -> np.ndarray:
    out = gray.copy()
    h, w = out.shape
    num_boxes = int(rng.integers(1, 2 if not hard else 4))
    for _ in range(num_boxes):
        bw = int(rng.integers(max(12, w // 30), max(20, w // (8 if hard else 12))))
        bh = int(rng.integers(max(4, h // 12), max(6, h // (4 if hard else 6))))
        x = int(rng.integers(0, max(1, w - bw)))
        y = int(rng.integers(0, max(1, h - bh)))
        color = int(rng.integers(180, 255))
        cv2.rectangle(out, (x, y), (x + bw, y + bh), color, -1)
    return out

def random_motion_blur(gray: np.ndarray, rng: np.random.Generator, hard: bool) -> np.ndarray:
    k = int(rng.choice([3, 5, 7] if not hard else [5, 7, 9]))
    kernel = np.zeros((k, k), dtype=np.float32)
    if rng.random() < 0.5:
        kernel[k // 2, :] = 1.0
    else:
        kernel[:, k // 2] = 1.0
    kernel /= kernel.sum()
    return cv2.filter2D(gray, -1, kernel)

def random_perspective(gray: np.ndarray, rng: np.random.Generator, hard: bool) -> np.ndarray:
    h, w = gray.shape
    jitter_x = max(2, int(w * (0.03 if not hard else 0.06)))
    jitter_y = max(2, int(h * (0.08 if not hard else 0.14)))
    src = np.float32([[0, 0], [w - 1, 0], [0, h - 1], [w - 1, h - 1]])
    dst = src + np.float32([
        [rng.integers(-jitter_x, jitter_x + 1), rng.integers(-jitter_y, jitter_y + 1)],
        [rng.integers(-jitter_x, jitter_x + 1), rng.integers(-jitter_y, jitter_y + 1)],
        [rng.integers(-jitter_x, jitter_x + 1), rng.integers(-jitter_y, jitter_y + 1)],
        [rng.integers(-jitter_x, jitter_x + 1), rng.integers(-jitter_y, jitter_y + 1)],
    ]).astype(np.float32)
    matrix = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(gray, matrix, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=255)

def down_up_scale(gray: np.ndarray, rng: np.random.Generator, hard: bool) -> np.ndarray:
    h, w = gray.shape
    factor = rng.uniform(0.55, 0.85) if not hard else rng.uniform(0.35, 0.70)
    nw = max(32, int(w * factor))
    nh = max(24, int(h * factor))
    small = cv2.resize(gray, (nw, nh), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)

def degrade_raw_image(gray: np.ndarray, rng: np.random.Generator, mode: str) -> np.ndarray:
    hard = mode == "hard_eval"
    img = gray.copy()

    if rng.random() < (0.30 if not hard else 0.55):
        img = random_perspective(img, rng, hard)

    if rng.random() < (0.85 if not hard else 0.95):
        alpha = float(rng.uniform(0.60, 1.35) if not hard else rng.uniform(0.45, 1.45))
        beta = float(rng.uniform(-45, 25) if not hard else rng.uniform(-70, 35))
        img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

    if rng.random() < (0.50 if not hard else 0.80):
        gamma = float(rng.uniform(0.70, 1.60) if not hard else rng.uniform(0.55, 2.10))
        table = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)]).astype(np.uint8)
        img = cv2.LUT(img, table)

    if rng.random() < (0.55 if not hard else 0.90):
        if rng.random() < 0.5:
            k = int(rng.choice([3, 5] if not hard else [5, 7, 9]))
            img = cv2.GaussianBlur(img, (k, k), sigmaX=float(rng.uniform(0.5, 1.8) if not hard else rng.uniform(1.0, 3.0)))
        else:
            img = random_motion_blur(img, rng, hard)

    if rng.random() < (0.55 if not hard else 0.85):
        noise = rng.normal(0, float(rng.uniform(4, 18) if not hard else rng.uniform(10, 28)), size=img.shape)
        img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    if rng.random() < (0.30 if not hard else 0.55):
        img = down_up_scale(img, rng, hard)

    if rng.random() < (0.35 if not hard else 0.60):
        img = add_shadow(img, rng, hard)

    if rng.random() < (0.18 if not hard else 0.45):
        img = add_glare(img, rng, hard)

    if rng.random() < (0.22 if not hard else 0.50):
        img = add_stains(img, rng, hard)

    if rng.random() < (0.10 if not hard else 0.35):
        img = add_occlusion(img, rng, hard)

    if rng.random() < (0.20 if not hard else 0.40):
        quality = int(rng.integers(25, 65) if not hard else rng.integers(18, 45))
        ok, enc = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
        if ok:
            img = cv2.imdecode(enc, cv2.IMREAD_GRAYSCALE)

    return img

def remove_isolated_pixels(binary_inv: np.ndarray) -> np.ndarray:
    binary_map = (binary_inv > 0).astype(np.uint8)
    neighbors = cv2.filter2D(binary_map, -1, np.ones((3, 3), np.uint8))
    isolated = (binary_map == 1) & (neighbors <= 2)
    cleaned = binary_inv.copy()
    cleaned[isolated] = 0
    return cleaned

def preprocess_for_legacy_crnn(gray: np.ndarray, target_height: int, max_width: int, min_width: int, required_width: int | None, augment_mode: str, rng: np.random.Generator):
    img = gray.copy()
    if augment_mode in {"train", "hard_eval"}:
        img = degrade_raw_image(img, rng, augment_mode)

    h, w = img.shape
    new_w = int(round(w * target_height / max(1, h)))
    new_w = max(min_width, min(max_width, new_w))
    interpolation = cv2.INTER_AREA if new_w <= w else cv2.INTER_CUBIC
    img = cv2.resize(img, (new_w, target_height), interpolation=interpolation)

    background = cv2.medianBlur(img, 31)
    img = cv2.divide(img, background, scale=255)
    max_val = max(1, int(np.max(img)))
    img = np.where(img >= 0.90 * max_val, 255, img).astype(np.uint8)
    img = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img)
    max_val = max(1, int(np.max(img)))
    img = np.where(img >= 0.75 * max_val, 255, img).astype(np.uint8)

    binary_inv = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 10)
    binary_inv = remove_isolated_pixels(binary_inv)

    if augment_mode in {"train", "hard_eval"} and rng.random() < (0.30 if augment_mode == "train" else 0.50):
        kernel = np.ones((2, 2), np.uint8)
        binary_inv = cv2.dilate(binary_inv, kernel, iterations=1) if rng.random() < 0.5 else cv2.erode(binary_inv, kernel, iterations=1)

    final_width = max(new_w, int(required_width) if required_width is not None else new_w)
    final_width = max(min_width, min(max_width, final_width))
    image = (binary_inv.astype(np.float32) / 255.0)[..., None]
    return image, new_w, final_width


PROFILE_NAMES = tuple(CFG["preprocess_profiles"])

def apply_preprocess_profile(gray: np.ndarray, profile: str) -> np.ndarray:
    img = gray.copy()
    if profile == "legacy_clean":
        return img
    if profile == "mild_denoise":
        img = cv2.bilateralFilter(img, 5, 30, 30)
        img = cv2.fastNlMeansDenoising(img, None, h=5, templateWindowSize=7, searchWindowSize=21)
        return img
    if profile == "mild_sharpen":
        blur = cv2.GaussianBlur(img, (0, 0), 1.2)
        img = cv2.addWeighted(img, 1.35, blur, -0.35, 0)
        return np.clip(img, 0, 255).astype(np.uint8)
    raise ValueError(f"Unknown preprocess_profile: {profile}")

def apply_postprocess_profile(binary_inv: np.ndarray, profile: str) -> np.ndarray:
    out = binary_inv.copy()
    if profile == "mild_denoise":
        kernel = np.ones((2, 2), np.uint8)
        out = cv2.morphologyEx(out, cv2.MORPH_OPEN, kernel, iterations=1)
    elif profile == "mild_sharpen":
        kernel = np.ones((2, 2), np.uint8)
        out = cv2.dilate(out, kernel, iterations=1)
    return out

def preprocess_for_legacy_crnn(
    gray: np.ndarray,
    target_height: int,
    max_width: int,
    min_width: int,
    required_width: int | None,
    augment_mode: str,
    rng: np.random.Generator,
    preprocess_profile: str = "legacy_clean",
):
    img = gray.copy()
    if augment_mode in {"train", "hard_eval"}:
        img = degrade_raw_image(img, rng, augment_mode)

    img = apply_preprocess_profile(img, preprocess_profile)

    h, w = img.shape
    new_w = int(round(w * target_height / max(1, h)))
    new_w = max(min_width, min(max_width, new_w))
    interpolation = cv2.INTER_AREA if new_w <= w else cv2.INTER_CUBIC
    img = cv2.resize(img, (new_w, target_height), interpolation=interpolation)

    background = cv2.medianBlur(img, 31)
    img = cv2.divide(img, background, scale=255)
    max_val = max(1, int(np.max(img)))
    img = np.where(img >= 0.90 * max_val, 255, img).astype(np.uint8)
    img = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img)
    max_val = max(1, int(np.max(img)))
    img = np.where(img >= 0.75 * max_val, 255, img).astype(np.uint8)

    binary_inv = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 10)
    binary_inv = remove_isolated_pixels(binary_inv)
    binary_inv = apply_postprocess_profile(binary_inv, preprocess_profile)

    if augment_mode in {"train", "hard_eval"} and rng.random() < (0.30 if augment_mode == "train" else 0.50):
        kernel = np.ones((2, 2), np.uint8)
        binary_inv = cv2.dilate(binary_inv, kernel, iterations=1) if rng.random() < 0.5 else cv2.erode(binary_inv, kernel, iterations=1)

    final_width = max(new_w, int(required_width) if required_width is not None else new_w)
    final_width = max(min_width, min(max_width, final_width))
    image = (binary_inv.astype(np.float32) / 255.0)[..., None]
    return image, new_w, final_width


In [7]:
class OCRSequence(tf.keras.utils.Sequence):
    def __init__(
        self,
        df: pd.DataFrame,
        batch_size: int,
        max_label_len: int,
        target_height: int,
        downsample_factor: int,
        max_width: int,
        min_width: int,
        width_pad_multiple: int,
        fixed_width: int,
        preprocess_profile: str = "legacy_clean",
        augment_mode: str = "none",
        shuffle: bool = False,
    ):
        self.df = df.reset_index(drop=True).copy()
        self.batch_size = batch_size
        self.max_label_len = max_label_len
        self.target_height = target_height
        self.downsample_factor = downsample_factor
        self.max_width = max_width
        self.min_width = min_width
        self.width_pad_multiple = width_pad_multiple
        self.fixed_width = int(fixed_width)
        self.preprocess_profile = preprocess_profile
        self.augment_mode = augment_mode
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        self.batches = []
        self.on_epoch_end()

        min_needed = int(np.ceil(self.df["required_width"].max() / self.width_pad_multiple) * self.width_pad_multiple)
        if self.fixed_width < min_needed:
            raise ValueError(f"fixed_width={self.fixed_width} nho hon required_width toi thieu {min_needed}")

    def __len__(self):
        return len(self.batches)

    def on_epoch_end(self):
        order = np.argsort(self.df["final_width_hint"].to_numpy())
        if self.shuffle:
            bucket_size = max(self.batch_size * 16, self.batch_size)
            buckets = [order[i:i + bucket_size] for i in range(0, len(order), bucket_size)]
            random.shuffle(buckets)
            for bucket in buckets:
                np.random.shuffle(bucket)
            order = np.concatenate(buckets) if buckets else np.array([], dtype=np.int64)
        self.indices = order
        self.batches = [self.indices[i:i + self.batch_size] for i in range(0, len(self.indices), self.batch_size)]

    def get_batch_records(self, batch_idx: int) -> pd.DataFrame:
        return self.df.iloc[self.batches[batch_idx]].reset_index(drop=True)

    def __getitem__(self, batch_idx):
        batch_df = self.get_batch_records(batch_idx)
        images, encoded_labels, label_lengths = [], [], []

        for row in batch_df.itertuples():
            gray = cv2.imread(row.path, cv2.IMREAD_GRAYSCALE)
            if gray is None:
                raise FileNotFoundError(f"Cannot read image: {row.path}")

            if self.augment_mode == "hard_eval":
                rng = np.random.default_rng(stable_int_seed(row.filename))
            elif self.augment_mode == "train":
                rng = np.random.default_rng(SEED + batch_idx * 1000 + int(row.Index))
            else:
                rng = np.random.default_rng(0)

            image, _, _ = preprocess_for_legacy_crnn(
                gray=gray,
                target_height=self.target_height,
                max_width=self.max_width,
                min_width=self.min_width,
                required_width=row.required_width,
                augment_mode=self.augment_mode,
                rng=rng,
                preprocess_profile=self.preprocess_profile,
            )
            images.append(image)
            encoded_labels.append(row.encoded)
            label_lengths.append(row.encoded_length)

        batch_images = np.zeros((len(images), self.target_height, self.fixed_width, 1), dtype=np.float32)
        batch_labels = np.zeros((len(images), self.max_label_len), dtype=np.int32)

        for i, (img, label) in enumerate(zip(images, encoded_labels)):
            width = min(img.shape[1], self.fixed_width)
            batch_images[i, :, :width, :] = img[:, :width, :]
            batch_labels[i, :len(label)] = np.array(label, dtype=np.int32)

        input_length = np.full((len(images), 1), self.fixed_width // self.downsample_factor, dtype=np.int32)
        label_length = np.array(label_lengths, dtype=np.int32).reshape(-1, 1)

        inputs = {
            "input_image": batch_images,
            "the_labels": batch_labels,
            "input_length": input_length,
            "label_length": label_length,
        }
        outputs = np.zeros((len(images),), dtype=np.float32)
        return inputs, outputs

def build_eval_df_from_root(root: Path, prefix: str):
    labels_path = root / "labels.json"
    image_dir = root / "data"
    with open(labels_path, "r", encoding="utf-8") as f:
        labels = json.load(f)
    rows = []
    for filename, text in labels.items():
        image_path = image_dir / filename
        if not image_path.exists():
            continue
        try:
            with Image.open(image_path) as img:
                width, height = img.size
        except Exception:
            continue
        scaled_width = int(round(width * CFG["target_height"] / max(1, height)))
        scaled_width = max(CFG["min_width"], min(CFG["max_width"], scaled_width))
        rows.append(
            {
                "dataset_root": str(root),
                "filename": filename,
                "path": str(image_path),
                "text": text,
                "label_length": len(text),
                "orig_width": width,
                "orig_height": height,
                "scaled_width": scaled_width,
                "source_group": f"{prefix}:{Path(filename).stem}",
            }
        )
    out = pd.DataFrame(rows)
    out["required_width"] = (out["label_length"] + 2) * CFG["downsample_factor"]
    out["encoded"] = out["text"].apply(encode_text)
    out["encoded_length"] = out["encoded"].apply(len)
    out["final_width_hint"] = out[["scaled_width", "required_width"]].max(axis=1)
    return out.reset_index(drop=True)

def build_camera_hard_df():
    if CAMERA_HARD_ROOT is not None:
        return build_eval_df_from_root(CAMERA_HARD_ROOT, "camera"), "real_camera"

    sample_size = min(CFG["camera_hard_sample_size"], len(val_df))
    hard_df = val_df.sample(sample_size, random_state=SEED).reset_index(drop=True).copy()
    return hard_df, "synthetic_hard"

camera_hard_df, camera_hard_mode = build_camera_hard_df()
print("camera_hard_mode:", camera_hard_mode, "| samples:", len(camera_hard_df))


camera_hard_mode: synthetic_hard | samples: 256


In [8]:
def decode_batch_predictions(predictions, input_length, decoder="greedy", beam_width=10):
    use_greedy = decoder == "greedy"
    decoded, _ = K.ctc_decode(predictions, input_length=input_length, greedy=use_greedy, beam_width=beam_width, top_paths=1)
    decoded = decoded[0].numpy()

    texts, confidences = [], []
    blank_idx = len(char_list)
    for i, seq in enumerate(decoded):
        tokens = [int(tok) for tok in seq if int(tok) != -1 and int(tok) != blank_idx]
        texts.append("".join(idx_to_char[tok] for tok in tokens))
        timestep_probs = np.max(predictions[i, : int(input_length[i]), :], axis=-1)
        timestep_probs = np.clip(timestep_probs, 1e-6, 1.0)
        conf = float(np.exp(np.mean(np.log(timestep_probs)))) if len(timestep_probs) else 0.0
        confidences.append(conf)
    return texts, confidences

def compute_ocr_metrics(pred_texts, gt_texts):
    cer_scores, wer_scores, ser_scores = [], [], []
    for pred, gt in zip(pred_texts, gt_texts):
        cer_scores.append(editdistance.eval(list(pred.lower()), list(gt.lower())) / max(1, len(gt)))
        wer_scores.append(editdistance.eval(pred.lower().split(), gt.lower().split()) / max(1, len(gt.split())))
        ser_scores.append(float(pred.strip() != gt.strip()))
    return {
        "cer": float(np.mean(cer_scores)) if cer_scores else 1.0,
        "wer": float(np.mean(wer_scores)) if wer_scores else 1.0,
        "ser": float(np.mean(ser_scores)) if ser_scores else 1.0,
        "exact_match_case_sensitive": float(np.mean([p.strip() == g.strip() for p, g in zip(pred_texts, gt_texts)])) if gt_texts else 0.0,
        "exact_match_case_insensitive": float(np.mean([p.strip().lower() == g.strip().lower() for p, g in zip(pred_texts, gt_texts)])) if gt_texts else 0.0,
    }

def ctc_lambda_func(args):
    y_pred, labels, input_length, label_length = args
    return K.ctc_batch_cost(labels, y_pred, input_length, label_length)

def build_crnn_ctc_model(num_chars: int, max_label_len: int):
    inputs = layers.Input(shape=(CFG["target_height"], None, 1), name="input_image")

    x = layers.Conv2D(64, (3, 3), padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPool2D(pool_size=(3, 2))(x)

    x = layers.Conv2D(128, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPool2D(pool_size=(2, 2))(x)

    x = layers.Conv2D(256, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.MaxPool2D(pool_size=(2, 1))(x)

    x = layers.Conv2D(256, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.MaxPool2D(pool_size=(2, 1))(x)

    x = layers.Conv2D(512, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPool2D(pool_size=(3, 1))(x)

    x = layers.Lambda(lambda t: tf.reshape(t, [tf.shape(t)[0], tf.shape(t)[2], tf.shape(t)[1] * tf.shape(t)[3]]))(x)
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True, dropout=0.25))(x)
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True, dropout=0.25))(x)
    outputs = layers.Dense(num_chars + 1, activation="softmax", dtype="float32")(x)

    inference_model = keras.Model(inputs=inputs, outputs=outputs, name="ocr_inference_model")

    labels = layers.Input(name="the_labels", shape=[max_label_len], dtype="int32")
    input_length = layers.Input(name="input_length", shape=[1], dtype="int32")
    label_length = layers.Input(name="label_length", shape=[1], dtype="int32")
    loss_out = layers.Lambda(ctc_lambda_func, name="ctc")([outputs, labels, input_length, label_length])
    training_model = keras.Model(inputs=[inputs, labels, input_length, label_length], outputs=loss_out, name="ocr_training_model")
    return inference_model, training_model

runtime_state = {"completed_epochs": 0, "resume_source": None}
max_label_len = int(records_df["encoded_length"].max())
num_chars = len(char_list)

tf.keras.backend.clear_session()

with strategy.scope():
    act_model, train_model = build_crnn_ctc_model(num_chars=num_chars, max_label_len=max_label_len)
    optimizer = keras.optimizers.Adam(learning_rate=CFG["initial_lr"], clipnorm=5.0)
    train_model.compile(loss={"ctc": lambda y_true, y_pred: y_pred}, optimizer=optimizer)

restored_from = None
if WEIGHTS_PATH is not None and Path(WEIGHTS_PATH).exists():
    act_model.load_weights(str(WEIGHTS_PATH))
    restored_from = str(WEIGHTS_PATH)

runtime_state["resume_source"] = restored_from
print("Restored from:", restored_from)


Restored from: /kaggle/input/models/trinhminhkhoak18hcm/ocr-train-chuan-checkpoint-weights/gguf/default/1/ocr-train-chuan_checkpoint.weights.h5


In [9]:
def evaluate_sequence(model, sequence, tag: str, save_files: bool = True, decoder: str = "greedy", beam_width: int = 10):
    rows = []
    total_pred_time = 0.0
    total_lines = 0

    for batch_idx in range(len(sequence)):
        batch_inputs, _ = sequence[batch_idx]
        batch_records = sequence.get_batch_records(batch_idx)
        images = batch_inputs["input_image"]
        input_length = batch_inputs["input_length"][:, 0]

        start = time.perf_counter()
        batch_pred = model.predict(images, verbose=0)
        total_pred_time += time.perf_counter() - start
        total_lines += len(images)

        pred_texts, confidences = decode_batch_predictions(
            batch_pred,
            input_length=input_length,
            decoder=decoder,
            beam_width=beam_width,
        )

        for row, pred_text, conf in zip(batch_records.itertuples(), pred_texts, confidences):
            gt = row.text
            rows.append(
                {
                    "filename": row.filename,
                    "path": row.path,
                    "ground_truth": gt,
                    "prediction": pred_text,
                    "confidence": conf,
                    "char_distance_lower": editdistance.eval(list(pred_text.lower()), list(gt.lower())),
                    "word_distance_lower": editdistance.eval(pred_text.lower().split(), gt.lower().split()),
                    "exact_match": int(pred_text.strip() == gt.strip()),
                }
            )

    results_df = pd.DataFrame(rows)
    metrics = compute_ocr_metrics(results_df["prediction"].tolist(), results_df["ground_truth"].tolist())
    metrics["char_accuracy"] = float(1.0 - metrics["cer"])
    metrics["avg_confidence"] = float(results_df["confidence"].mean()) if not results_df.empty else 0.0
    metrics["line_fps_batch_throughput"] = float(total_lines / max(total_pred_time, 1e-9))
    metrics["avg_line_latency_ms_batch"] = float((total_pred_time / max(total_lines, 1)) * 1000.0)
    metrics["num_eval_samples"] = int(len(results_df))
    metrics["tag"] = tag
    metrics["decoder"] = decoder
    metrics["beam_width"] = int(beam_width)

    pred_path = None
    metrics_path = None
    if save_files:
        pred_path = PATHS["predictions_dir"] / f"{tag}_predictions.csv"
        metrics_path = PATHS["metrics_dir"] / f"{tag}_metrics.json"
        results_df.to_csv(pred_path, index=False, encoding="utf-8-sig")
        with open(metrics_path, "w", encoding="utf-8") as f:
            json.dump(metrics, f, ensure_ascii=False, indent=2)

    return results_df, metrics, pred_path, metrics_path


In [10]:
def resolve_fixed_width(df: pd.DataFrame, requested_width: int, seq_max_width: int) -> int:
    required_floor = int(np.ceil(df["required_width"].max() / CFG["width_pad_multiple"]) * CFG["width_pad_multiple"])
    width = int(np.ceil(int(requested_width) / CFG["width_pad_multiple"]) * CFG["width_pad_multiple"])
    width = max(required_floor, width)
    width = min(seq_max_width, max(CFG["min_width"], width))
    return width

def build_eval_sequence(
    df: pd.DataFrame,
    eval_max_width: int,
    preprocess_profile: str = "legacy_clean",
    augment_mode: str = "none",
):
    eval_max_width = int(np.ceil(eval_max_width / CFG["width_pad_multiple"]) * CFG["width_pad_multiple"])
    fixed_width = resolve_fixed_width(df, eval_max_width, eval_max_width)
    seq = OCRSequence(
        df,
        batch_size=CFG["global_batch_size"],
        max_label_len=max_label_len,
        target_height=CFG["target_height"],
        downsample_factor=CFG["downsample_factor"],
        max_width=eval_max_width,
        min_width=CFG["min_width"],
        width_pad_multiple=CFG["width_pad_multiple"],
        fixed_width=fixed_width,
        preprocess_profile=preprocess_profile,
        augment_mode=augment_mode,
        shuffle=False,
    )
    return fixed_width, seq

RAW_BASELINE_WIDTH = DEFAULT_BACKBONE_WIDTH
RAW_BASELINE_DECODER = DEFAULT_BACKBONE_DECODER
RAW_BASELINE_BEAM = DEFAULT_BACKBONE_BEAM

DEFAULT_EVAL_FIXED_WIDTH, val_seq = build_eval_sequence(val_df, RAW_BASELINE_WIDTH, preprocess_profile="legacy_clean", augment_mode="none")
_, camera_hard_seq = build_eval_sequence(
    camera_hard_df,
    RAW_BASELINE_WIDTH,
    preprocess_profile="legacy_clean",
    augment_mode="none" if camera_hard_mode == "real_camera" else "hard_eval",
)

print("DEFAULT_EVAL_FIXED_WIDTH:", DEFAULT_EVAL_FIXED_WIDTH)
print("Val batch image shape:", val_seq[0][0]["input_image"].shape)
print("Raw baseline config:", RAW_BASELINE_WIDTH, RAW_BASELINE_DECODER, RAW_BASELINE_BEAM)
act_model.summary()

baseline_val_df, baseline_val_metrics, _, _ = evaluate_sequence(
    act_model,
    val_seq,
    "baseline_raw_grouped_val",
    save_files=True,
    decoder=RAW_BASELINE_DECODER,
    beam_width=RAW_BASELINE_BEAM,
)
baseline_hard_df, baseline_hard_metrics, _, _ = evaluate_sequence(
    act_model,
    camera_hard_seq,
    "baseline_raw_camera_hard",
    save_files=True,
    decoder=RAW_BASELINE_DECODER,
    beam_width=RAW_BASELINE_BEAM,
)

print("Baseline raw grouped-val metrics:")
display(pd.DataFrame([baseline_val_metrics]).T.rename(columns={0: "value"}))
print("Baseline raw hard metrics:")
display(pd.DataFrame([baseline_hard_metrics]).T.rename(columns={0: "value"}))


DEFAULT_EVAL_FIXED_WIDTH: 2304
Val batch image shape: (4, 96, 2304, 1)
Raw baseline config: 2304 greedy 10


Model: "ocr_inference_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 96, None, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 96, None, 64)   │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 96, None, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 96, None, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, None, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, None, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, None, 128)  │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 32, None, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, None, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, None, 256)  │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, None, 256)  │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 16, None, 256)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, None, 256)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, None, 256)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 8, None, 256)   │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 8, None, 256)   │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 8, None, 256)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, None, 256)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 4, None, 256)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 4, None, 512)   │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 4, None, 512)   │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 4, None, 512)   │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 5,407,965 (20.63 MB)

 Trainable params: 5,405,533 (20.62 MB)

...

I0000 00:00:1774171759.799341      73 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1774171759.799346      71 cuda_dnn.cc:529] Loaded cuDNN version 91002


Baseline raw grouped-val metrics:


,value
cer,0.019918
wer,0.053734
ser,0.274376
exact_match_case_sensitive,0.725624
exact_match_case_insensitive,0.741497
char_accuracy,0.980082
avg_confidence,0.98786
line_fps_batch_throughput,9.509544
avg_line_latency_ms_batch,105.157512
num_eval_samples,1323


Baseline raw hard metrics:


,value
cer,0.562548
wer,0.883919
ser,0.886719
exact_match_case_sensitive,0.113281
exact_match_case_insensitive,0.113281
char_accuracy,0.437452
avg_confidence,0.960879
line_fps_batch_throughput,9.695937
avg_line_latency_ms_batch,103.135985
num_eval_samples,256


In [11]:
TOKEN_PATTERN = re.compile(r"\s+|\w+|[^\w\s]", re.UNICODE)

STATIC_CONFUSIONS = {
    "a": ["ạ", "á", "à", "ả", "ã", "ă", "â"],
    "ă": ["a", "â"],
    "â": ["a", "ă"],
    "e": ["é", "è", "ẻ", "ẽ", "ẹ", "ê"],
    "ê": ["e", "ề", "ế", "ể", "ễ", "ệ"],
    "i": ["í", "ì", "ỉ", "ĩ", "ị"],
    "o": ["ó", "ò", "ỏ", "õ", "ọ", "ô", "ơ"],
    "ô": ["o", "ồ", "ố", "ổ", "ỗ", "ộ"],
    "ơ": ["o", "ờ", "ớ", "ở", "ỡ", "ợ"],
    "u": ["ú", "ù", "ủ", "ũ", "ụ", "ư", "n"],
    "ư": ["u", "ừ", "ứ", "ử", "ữ", "ự"],
    "y": ["ý", "ỳ", "ỷ", "ỹ", "ỵ"],
    "m": ["n", "rn"],
    "n": ["m", "u"],
}

def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip()

def tokenize_text(text: str):
    return TOKEN_PATTERN.findall(text)

def is_word_token(token: str) -> bool:
    return bool(token) and (not token.isspace()) and any(ch.isalpha() for ch in token)

def strip_diacritics(text: str) -> str:
    normalized = unicodedata.normalize("NFD", text)
    return "".join(ch for ch in normalized if unicodedata.category(ch) != "Mn")

def digit_signature(text: str) -> str:
    return "".join(ch for ch in str(text or "") if ch.isdigit())

def dominant_case_form(token_lower: str):
    forms = TOKEN_CASE_COUNTER.get(token_lower)
    if not forms:
        return None
    return forms.most_common(1)[0][0]

def adapt_case(candidate: str, original: str) -> str:
    if original.isupper():
        return candidate.upper()
    if original[:1].isupper():
        return candidate[:1].upper() + candidate[1:]
    return candidate.lower()

TRAIN_SENTENCES = train_df["text"].astype(str).map(normalize_text).tolist()
TOKEN_COUNTER = Counter()
TOKEN_CASE_COUNTER = defaultdict(Counter)
CHAR_5GRAM_COUNTER = Counter()

for text in TRAIN_SENTENCES:
    lowered = text.lower()
    padded = f"^^{lowered}$$"
    CHAR_5GRAM_COUNTER.update([padded[i:i + 5] for i in range(max(0, len(padded) - 4))])
    for tok in tokenize_text(text):
        if is_word_token(tok):
            TOKEN_COUNTER[tok.lower()] += 1
            TOKEN_CASE_COUNTER[tok.lower()][tok] += 1

def char_lm_score(text: str) -> float:
    text = normalize_text(text).lower()
    padded = f"^^{text}$$"
    grams = [padded[i:i + 5] for i in range(max(0, len(padded) - 4))]
    if not grams:
        return -1.0
    return float(np.mean([math.log1p(CHAR_5GRAM_COUNTER.get(g, 0)) for g in grams]) / 3.0)

def word_lm_score(text: str) -> float:
    tokens = [tok.lower() for tok in tokenize_text(text) if is_word_token(tok)]
    if not tokens:
        return 0.0
    vals = [math.log1p(TOKEN_COUNTER.get(tok, 0)) / 4.0 if TOKEN_COUNTER.get(tok, 0) > 0 else -0.45 for tok in tokens]
    return float(np.mean(vals))

def total_lm_score(text: str) -> float:
    return 0.80 * char_lm_score(text) + 0.40 * word_lm_score(text)

LEXICON_LOWER = sorted(
    token for token, freq in TOKEN_COUNTER.items()
    if freq >= 1 and 2 <= len(token) <= CFG["token_corrector_max_len"] and any(ch.isalpha() for ch in token)
)

TOKEN_BUCKETS = defaultdict(list)
for token in LEXICON_LOWER:
    key = (strip_diacritics(token)[:1], len(token))
    TOKEN_BUCKETS[key].append(token)

def build_token_vocab(tokens):
    charset = sorted(set("".join(tokens)))
    specials = ["<pad>", "<eos>"]
    vocab = specials + charset
    char_to_id = {ch: idx for idx, ch in enumerate(vocab)}
    id_to_char = {idx: ch for ch, idx in char_to_id.items()}
    return vocab, char_to_id, id_to_char

TEXT_VOCAB, TEXT_CHAR_TO_ID, TEXT_ID_TO_CHAR = build_token_vocab(LEXICON_LOWER + [tok.lower() for tok in val_df["text"].astype(str).str.replace(r"\s+", "", regex=True).tolist()[:10]])
PAD_ID = TEXT_CHAR_TO_ID["<pad>"]
EOS_ID = TEXT_CHAR_TO_ID["<eos>"]

def encode_token_ids(token: str, max_len: int):
    token = normalize_text(token).lower()
    ids = [TEXT_CHAR_TO_ID[ch] for ch in token[: max_len - 1] if ch in TEXT_CHAR_TO_ID]
    ids.append(EOS_ID)
    return ids

def decode_token_ids(ids):
    chars = []
    for idx in ids:
        idx = int(idx)
        if idx == EOS_ID:
            break
        if idx == PAD_ID:
            continue
        chars.append(TEXT_ID_TO_CHAR.get(idx, ""))
    return "".join(chars)

def apply_token_noise(token: str, rng: np.random.Generator) -> str:
    chars = list(token.lower())
    if not chars:
        return token.lower()
    num_ops = 1 if len(chars) <= 6 else 2
    for _ in range(num_ops):
        op = rng.choice(["confusion", "strip_diacritic", "delete_char", "duplicate_char", "swap_adjacent"])
        if op == "confusion":
            positions = [i for i, ch in enumerate(chars) if ch in STATIC_CONFUSIONS]
            if positions:
                pos = int(rng.choice(positions))
                chars[pos] = rng.choice(STATIC_CONFUSIONS[chars[pos]])
        elif op == "strip_diacritic":
            positions = [i for i, ch in enumerate(chars) if strip_diacritics(ch) != ch]
            if positions:
                pos = int(rng.choice(positions))
                chars[pos] = strip_diacritics(chars[pos])
        elif op == "delete_char":
            positions = [i for i, ch in enumerate(chars) if ch.isalpha()]
            if len(chars) > 2 and positions:
                del chars[int(rng.choice(positions))]
        elif op == "duplicate_char":
            positions = [i for i, ch in enumerate(chars) if ch.isalpha()]
            if positions and len(chars) < CFG["token_corrector_max_len"] - 2:
                pos = int(rng.choice(positions))
                chars.insert(pos, chars[pos])
        elif op == "swap_adjacent":
            if len(chars) >= 2:
                pos = int(rng.integers(0, len(chars) - 1))
                chars[pos], chars[pos + 1] = chars[pos + 1], chars[pos]
    return "".join(chars) or token.lower()

weighted_tokens = []
for token, freq in TOKEN_COUNTER.items():
    if 2 <= len(token) <= CFG["token_corrector_max_len"] and any(ch.isalpha() for ch in token):
        repeat_factor = max(1, min(5, int(math.log2(freq + 1)) + 1))
        weighted_tokens.extend([token.lower()] * repeat_factor)

rng_perm = np.random.default_rng(SEED)
perm = rng_perm.permutation(len(weighted_tokens))
split_at = int(len(perm) * (1.0 - CFG["token_corrector_val_ratio"]))
corrector_train_tokens = [weighted_tokens[i] for i in perm[:split_at]]
corrector_val_tokens = [weighted_tokens[i] for i in perm[split_at:]]

def make_token_pairs(tokens, repeats: int, seed_offset: int):
    pairs = []
    for idx, token in enumerate(tokens):
        for rep in range(repeats):
            rng = np.random.default_rng(SEED + seed_offset + idx * 31 + rep * 101)
            noisy = apply_token_noise(token, rng)
            pairs.append((noisy, token))
    return pairs

train_pairs = make_token_pairs(corrector_train_tokens, CFG["token_corrector_train_repeats"], 0)
val_pairs = make_token_pairs(corrector_val_tokens[: max(1, len(corrector_val_tokens) // 2)], 1, 999_999)

TOKEN_MAX_LEN = min(
    CFG["token_corrector_max_len"],
    max(max(len(src) for src, _ in train_pairs + val_pairs), max(len(t) for t in LEXICON_LOWER)) + 2,
)

def build_token_arrays(pairs):
    enc_inputs = np.full((len(pairs), TOKEN_MAX_LEN), PAD_ID, dtype=np.int32)
    targets = np.full((len(pairs), TOKEN_MAX_LEN), PAD_ID, dtype=np.int32)
    weights = np.zeros((len(pairs), TOKEN_MAX_LEN), dtype=np.float32)
    for i, (src, tgt) in enumerate(pairs):
        src_ids = encode_token_ids(src, TOKEN_MAX_LEN)
        tgt_ids = encode_token_ids(tgt, TOKEN_MAX_LEN)
        enc_inputs[i, : len(src_ids)] = src_ids
        targets[i, : len(tgt_ids)] = tgt_ids
        weights[i, : len(tgt_ids)] = 1.0
    return enc_inputs, targets, weights

enc_train, tgt_train, w_train = build_token_arrays(train_pairs)
enc_val, tgt_val, w_val = build_token_arrays(val_pairs)

TOKEN_CORRECTOR_CACHE = {}

text_asset_payload = {
    "train_pairs": len(train_pairs),
    "val_pairs": len(val_pairs),
    "token_vocab_size": len(TEXT_VOCAB),
    "token_max_len": int(TOKEN_MAX_LEN),
    "unique_tokens": len(LEXICON_LOWER),
    "weighted_tokens": len(weighted_tokens),
}
with open(PATHS["token_assets_json"], "w", encoding="utf-8") as f:
    json.dump(text_asset_payload, f, ensure_ascii=False, indent=2)
print(json.dumps(text_asset_payload, ensure_ascii=False, indent=2))


{
  "train_pairs": 45748,
  "val_pairs": 497,
  "token_vocab_size": 108,
  "token_max_len": 15,
  "unique_tokens": 3581,
  "weighted_tokens": 12432
}


In [12]:
def build_token_corrector_model(vocab_size: int, max_len: int):
    token_inputs = layers.Input(shape=(max_len,), dtype="int32", name="token_inputs")
    emb = layers.Embedding(vocab_size, CFG["token_corrector_embedding_dim"], mask_zero=False, dtype="float32", name="tok_emb")(token_inputs)
    x = layers.Bidirectional(
        layers.GRU(CFG["token_corrector_hidden_dim"], return_sequences=True, dtype="float32"),
        name="tok_bigru_1",
    )(emb)
    x = layers.Bidirectional(
        layers.GRU(CFG["token_corrector_hidden_dim"], return_sequences=True, dtype="float32"),
        name="tok_bigru_2",
    )(x)
    probs = layers.Dense(vocab_size, activation="softmax", dtype="float32", name="tok_output")(x)
    model = keras.Model(token_inputs, probs, name="token_corrector_model")
    dummy = np.zeros((1, max_len), dtype=np.int32)
    _ = model(dummy, training=False)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=2e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )
    return model

token_corrector_model = build_token_corrector_model(len(TEXT_VOCAB), TOKEN_MAX_LEN)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_acc", mode="max", patience=2, restore_best_weights=True),
]
history = token_corrector_model.fit(
    enc_train,
    tgt_train,
    sample_weight=w_train,
    validation_data=(enc_val, tgt_val, w_val),
    epochs=CFG["token_corrector_epochs"],
    batch_size=CFG["token_corrector_batch_size"],
    callbacks=callbacks,
    verbose=1,
)

pd.DataFrame(history.history).to_csv(PATHS["token_history_csv"], index=False, encoding="utf-8-sig")
display(pd.DataFrame(history.history))

def decode_corrected_token(token: str):
    token_key = token.lower()
    if token_key in TOKEN_CORRECTOR_CACHE:
        return TOKEN_CORRECTOR_CACHE[token_key]

    src_ids = np.full((1, TOKEN_MAX_LEN), PAD_ID, dtype=np.int32)
    encoded = encode_token_ids(token_key, TOKEN_MAX_LEN)
    src_ids[0, : len(encoded)] = encoded
    seq_probs = token_corrector_model(src_ids, training=False).numpy()[0]
    output_ids = []
    probs = []
    for step in seq_probs:
        next_id = int(np.argmax(step))
        prob = float(np.max(step))
        if next_id == EOS_ID:
            break
        if next_id == PAD_ID:
            continue
        output_ids.append(next_id)
        probs.append(prob)
    result = (decode_token_ids(output_ids), float(np.mean(probs)) if probs else 0.0)
    TOKEN_CORRECTOR_CACHE[token_key] = result
    return result

def preview_token_corrector_examples(pairs, limit=12):
    rows = []
    for noisy, target in pairs[:limit]:
        pred, avg_prob = decode_corrected_token(noisy)
        rows.append({"noisy": noisy, "target": target, "predicted": pred, "avg_prob": avg_prob})
    return pd.DataFrame(rows)

display(preview_token_corrector_examples(val_pairs, limit=12))


Epoch 1/20
179/179 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - acc: 0.1497 - loss: 0.6688 - val_acc: 0.2472 - val_loss: 0.2072
Epoch 2/20
179/179 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - acc: 0.2457 - loss: 0.1887 - val_acc: 0.2626 - val_loss: 0.1411
Epoch 3/20
179/179 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.2581 - loss: 0.1348 - val_acc: 0.2675 - val_loss: 0.1212
Epoch 4/20
179/179 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.2641 - loss: 0.1103 - val_acc: 0.2692 - val_loss: 0.1139
Epoch 5/20
179/179 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.2678 - loss: 0.0957 - val_acc: 0.2698 - val_loss: 0.1100
Epoch 6/20
179/179 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.2703 - loss: 0.0862 - val_acc: 0.2727 - val_loss: 0.1079
Epoch 7/20
179/179 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.2720 - loss: 0.0790 - val_acc: 0.2716 - val_loss: 0.1084
Epoch 8/20
179/179 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - acc: 0.2736 - loss: 0.0733 - val_acc: 0.2708 - val_loss: 0.1101


,acc,loss,val_acc,val_loss
0,0.194767,0.432329,0.247217,0.207182
1,0.250166,0.170907,0.262643,0.141051
2,0.260287,0.127354,0.267472,0.121201
3,0.265510,0.106070,0.269215,0.113891
4,0.268844,0.093021,0.269752,0.109981
5,0.271124,0.084181,0.272703,0.107887
6,0.272924,0.077453,0.271630,0.108369
7,0.274332,0.072037,0.270825,0.110080


,noisy,target,predicted,avg_prob
0,chenh,chênh,chenh,0.890945
1,sson,son,son,0.988749
2,huầ,hầu,hầu,0.913129
3,ln,len,lãn,0.660151
4,hcữ,chữ,chữ,0.993786
5,khúc,khúc,khúc,0.995732
6,thịềng,thiềng,thiềng,0.995604
7,prao,prao,prao,0.933168
8,cắắm,cắm,cắm,0.997095
9,nềim,niềm,niềm,0.920399


In [13]:
def lexical_candidates(token: str, limit: int = 5):
    token_lower = token.lower()
    key_base = strip_diacritics(token_lower)
    bucket = []
    for delta in (-1, 0, 1):
        bucket.extend(TOKEN_BUCKETS.get((key_base[:1], max(1, len(token_lower) + delta)), []))
    if not bucket:
        bucket = LEXICON_LOWER[:4000]
    dedup = list(dict.fromkeys(bucket))
    out = []
    if dedup:
        for candidate, score, _ in rf_process.extract(token_lower, dedup, scorer=rf_fuzz.ratio, limit=limit):
            if candidate == token_lower or score < CFG["token_similarity_threshold"]:
                continue
            out.append((candidate, float(score) / 100.0, "lexicon"))
    return out

def candidate_token_pool(base_token: str):
    base_lower = base_token.lower()
    candidates = []
    model_token, model_prob = decode_corrected_token(base_lower)
    if model_token and model_token != base_lower:
        candidates.append((model_token, model_prob, "model"))
    candidates.extend(lexical_candidates(base_token, limit=5))

    seen = set()
    dedup = []
    for token, score, source in candidates:
        token = token.lower()
        if token in seen:
            continue
        seen.add(token)
        dedup.append((token, float(score), source))
    return dedup

def accept_token_correction(base_token: str, candidate_token: str, sentence_before: str, sentence_after: str, support_score: float, source: str):
    base_lower = base_token.lower()
    cand_lower = candidate_token.lower()
    if not cand_lower or cand_lower == base_lower:
        return False, 0.0
    if digit_signature(base_token) != digit_signature(candidate_token):
        return False, 0.0
    if not any(ch.isalpha() for ch in cand_lower):
        return False, 0.0
    if TOKEN_COUNTER.get(cand_lower, 0) < CFG["token_min_freq"]:
        return False, 0.0

    token_ed = editdistance.eval(list(base_lower), list(cand_lower))
    max_ed = 1 if len(base_lower) <= 5 else 2 if len(base_lower) <= 10 else 3
    if token_ed > max_ed:
        return False, 0.0

    similarity = float(rf_fuzz.ratio(strip_diacritics(base_lower), strip_diacritics(cand_lower))) / 100.0
    if similarity < (CFG["token_similarity_threshold"] / 100.0) and strip_diacritics(base_lower) != strip_diacritics(cand_lower):
        return False, 0.0

    base_freq = TOKEN_COUNTER.get(base_lower, 0)
    cand_freq = TOKEN_COUNTER.get(cand_lower, 0)
    lm_delta = total_lm_score(sentence_after) - total_lm_score(sentence_before)

    if support_score < CFG["token_corrector_min_avg_prob"]:
        return False, lm_delta

    if CFG["token_same_lexicon_guard"] and base_freq >= CFG["token_min_freq"]:
        if cand_freq < max(base_freq * 4, base_freq + 8):
            return False, lm_delta
        if lm_delta < (CFG["token_corrector_accept_margin"] + 0.10):
            return False, lm_delta

    if source == "lexicon" and similarity < 0.80 and lm_delta < (CFG["token_corrector_accept_margin"] + 0.08):
        return False, lm_delta

    return lm_delta >= CFG["token_corrector_accept_margin"], lm_delta

def correct_sentence_tokens(text: str):
    parts = tokenize_text(text)
    word_positions = [idx for idx, part in enumerate(parts) if is_word_token(part)]
    candidates = []

    for pos in word_positions:
        base_token = parts[pos]
        base_lower = base_token.lower()
        base_freq = TOKEN_COUNTER.get(base_lower, 0)
        if base_freq >= CFG["token_min_freq"]:
            continue

        for cand_lower, support_score, source in candidate_token_pool(base_token):
            cand_token = adapt_case(cand_lower, base_token)
            new_parts = list(parts)
            new_parts[pos] = cand_token
            sentence_before = normalize_text("".join(parts))
            sentence_after = normalize_text("".join(new_parts))
            accepted, lm_delta = accept_token_correction(base_token, cand_token, sentence_before, sentence_after, support_score, source)
            if accepted:
                candidate_score = lm_delta + 0.15 * support_score + 0.03 * math.log1p(TOKEN_COUNTER.get(cand_lower, 0))
                candidates.append(
                    {
                        "position": pos,
                        "base_token": base_token,
                        "candidate_token": cand_token,
                        "source": source,
                        "support_score": float(support_score),
                        "lm_delta": float(lm_delta),
                        "candidate_score": float(candidate_score),
                    }
                )

    if not candidates:
        return normalize_text("".join(parts)), []

    candidates = sorted(candidates, key=lambda item: (-item["candidate_score"], -item["lm_delta"], -item["support_score"]))
    chosen = candidates[: CFG["max_token_corrections_per_line"]]
    new_parts = list(parts)
    for item in chosen:
        new_parts[item["position"]] = item["candidate_token"]
    return normalize_text("".join(new_parts)), chosen

def apply_token_corrector_to_df(base_df: pd.DataFrame, split_name: str):
    rows = []
    for row in base_df.itertuples():
        base_text = normalize_text(row.prediction)
        corrected_text, token_events = correct_sentence_tokens(base_text)
        rows.append(
            {
                "split": split_name,
                "filename": row.filename,
                "path": row.path,
                "ground_truth": row.ground_truth,
                "prediction": corrected_text,
                "base_prediction": base_text,
                "confidence": row.confidence,
                "accepted_correction": int(corrected_text != base_text),
                "num_token_corrections": int(len(token_events)),
                "token_events_json": json.dumps(token_events, ensure_ascii=False),
                "lm_base": total_lm_score(base_text),
                "lm_candidate": total_lm_score(corrected_text),
            }
        )
    return pd.DataFrame(rows)

def finalize_prediction_rows(pred_df: pd.DataFrame, tag: str):
    out = pred_df.copy()
    out["char_distance_lower"] = [editdistance.eval(list(p.lower()), list(g.lower())) for p, g in zip(out["prediction"], out["ground_truth"])]
    out["word_distance_lower"] = [editdistance.eval(p.lower().split(), g.lower().split()) for p, g in zip(out["prediction"], out["ground_truth"])]
    out["exact_match"] = (out["prediction"].str.strip() == out["ground_truth"].str.strip()).astype(int)
    metrics = compute_ocr_metrics(out["prediction"].tolist(), out["ground_truth"].tolist())
    metrics["char_accuracy"] = float(1.0 - metrics["cer"])
    metrics["avg_confidence"] = float(out["confidence"].mean()) if not out.empty else 0.0
    metrics["accepted_correction_ratio"] = float(out["accepted_correction"].mean()) if "accepted_correction" in out.columns and not out.empty else 0.0
    metrics["avg_token_corrections"] = float(out["num_token_corrections"].mean()) if "num_token_corrections" in out.columns and not out.empty else 0.0
    metrics["num_eval_samples"] = int(len(out))
    metrics["tag"] = tag
    return out, metrics

def save_stage_outputs(stage_df: pd.DataFrame, metrics: dict, tag: str):
    stage_df.to_csv(PATHS["predictions_dir"] / f"{tag}_predictions.csv", index=False, encoding="utf-8-sig")
    with open(PATHS["metrics_dir"] / f"{tag}_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

corrected_val_df = apply_token_corrector_to_df(baseline_val_df, "grouped_val")
corrected_hard_df = apply_token_corrector_to_df(baseline_hard_df, "camera_hard")
corrected_val_df, corrected_val_metrics = finalize_prediction_rows(corrected_val_df, "baseline_plus_token_corrector_grouped_val")
corrected_hard_df, corrected_hard_metrics = finalize_prediction_rows(corrected_hard_df, "baseline_plus_token_corrector_camera_hard")

save_stage_outputs(corrected_val_df, corrected_val_metrics, "baseline_plus_token_corrector_grouped_val")
save_stage_outputs(corrected_hard_df, corrected_hard_metrics, "baseline_plus_token_corrector_camera_hard")

compare_df = pd.DataFrame(
    [
        {"stage": "baseline_raw", "split": "grouped_val", **baseline_val_metrics},
        {"stage": "baseline_raw", "split": "camera_hard", **baseline_hard_metrics},
        {"stage": "baseline_plus_token_corrector", "split": "grouped_val", **corrected_val_metrics},
        {"stage": "baseline_plus_token_corrector", "split": "camera_hard", **corrected_hard_metrics},
    ]
)
compare_df.to_csv(PATHS["compare_csv"], index=False, encoding="utf-8-sig")
display(compare_df)

def compare_stage_to_baseline(baseline_df, stage_df, stage_name):
    merged = baseline_df[["filename", "prediction", "ground_truth"]].rename(columns={"prediction": "base_prediction"})
    merged = merged.merge(
        stage_df[["filename", "prediction", "accepted_correction", "num_token_corrections"]],
        on="filename",
        how="left",
    ).rename(columns={"prediction": "stage_prediction"})
    merged["base_exact"] = merged["base_prediction"].str.strip() == merged["ground_truth"].str.strip()
    merged["stage_exact"] = merged["stage_prediction"].str.strip() == merged["ground_truth"].str.strip()
    return {
        "stage": stage_name,
        "improved_vs_baseline": int(((~merged["base_exact"]) & merged["stage_exact"]).sum()),
        "hurt_vs_baseline": int((merged["base_exact"] & (~merged["stage_exact"])).sum()),
        "accepted_corrections": int(merged["accepted_correction"].fillna(0).sum()),
        "avg_token_corrections": float(merged["num_token_corrections"].fillna(0).mean()),
        "baseline_wrong": int((~merged["base_exact"]).sum()),
    }

stage_diagnostics = {
    "baseline_plus_token_corrector_grouped_val": compare_stage_to_baseline(baseline_val_df, corrected_val_df, "baseline_plus_token_corrector_grouped_val"),
    "baseline_plus_token_corrector_camera_hard": compare_stage_to_baseline(baseline_hard_df, corrected_hard_df, "baseline_plus_token_corrector_camera_hard"),
}
with open(PATHS["diagnostics_json"], "w", encoding="utf-8") as f:
    json.dump(stage_diagnostics, f, ensure_ascii=False, indent=2)

selected_stage = sorted(
    [
        {"stage": "baseline_raw", "grouped_df": baseline_val_df, "grouped_metrics": baseline_val_metrics, "hard_df": baseline_hard_df, "hard_metrics": baseline_hard_metrics},
        {"stage": "baseline_plus_token_corrector", "grouped_df": corrected_val_df, "grouped_metrics": corrected_val_metrics, "hard_df": corrected_hard_df, "hard_metrics": corrected_hard_metrics},
    ],
    key=lambda item: (
        -item["grouped_metrics"]["exact_match_case_sensitive"],
        item["grouped_metrics"]["cer"],
        item["grouped_metrics"]["wer"],
        item["hard_metrics"]["cer"],
    ),
)[0]

selected_payload = {
    "selected_stage": selected_stage["stage"],
    "backbone_weight_path": str(WEIGHTS_PATH) if WEIGHTS_PATH is not None else None,
    "token_corrector_model_path": str(PATHS["token_corrector_model"]),
    "raw_baseline_width": RAW_BASELINE_WIDTH,
    "raw_baseline_decoder": RAW_BASELINE_DECODER,
    "raw_baseline_beam_width": RAW_BASELINE_BEAM,
    "grouped_val_metrics": selected_stage["grouped_metrics"],
    "camera_hard_metrics": selected_stage["hard_metrics"],
    "baseline_grouped_val_metrics": baseline_val_metrics,
    "baseline_camera_hard_metrics": baseline_hard_metrics,
    "stage_diagnostics": stage_diagnostics,
}
with open(PATHS["selection_json"], "w", encoding="utf-8") as f:
    json.dump(selected_payload, f, ensure_ascii=False, indent=2)

act_model.load_weights(str(WEIGHTS_PATH))
act_model.save_weights(PATHS["selected_weights"])
act_model.save(PATHS["export_model"])
token_corrector_model.save(PATHS["token_corrector_model"])

def bundle_artifacts():
    if PATHS["bundle_zip"].exists():
        PATHS["bundle_zip"].unlink()
    bundle_root = OUTPUT_ROOT / "bundle_payload"
    if bundle_root.exists():
        shutil.rmtree(bundle_root)
    bundle_root.mkdir(parents=True, exist_ok=True)

    payload_paths = [
        PATHS["config_json"],
        PATHS["selection_json"],
        PATHS["compare_csv"],
        PATHS["token_history_csv"],
        PATHS["token_assets_json"],
        PATHS["diagnostics_json"],
        PATHS["selected_weights"],
        PATHS["token_corrector_model"],
    ]
    for src in payload_paths:
        if not src.exists():
            continue
        if src.is_dir():
            shutil.copytree(src, bundle_root / src.name, dirs_exist_ok=True)
        else:
            shutil.copy2(src, bundle_root / src.name)

    for directory_key in ["metrics_dir", "predictions_dir"]:
        src_dir = PATHS[directory_key]
        if src_dir.exists():
            shutil.copytree(src_dir, bundle_root / src_dir.name, dirs_exist_ok=True)

    if PATHS["export_model"].exists():
        dst_model = bundle_root / PATHS["export_model"].name
        if PATHS["export_model"].is_dir():
            shutil.copytree(PATHS["export_model"], dst_model, dirs_exist_ok=True)
        else:
            shutil.copy2(PATHS["export_model"], dst_model)

    shutil.make_archive(str(PATHS["bundle_zip"]).replace(".zip", ""), "zip", root_dir=bundle_root)
    return PATHS["bundle_zip"]

bundle_artifacts()
print(json.dumps(selected_payload, ensure_ascii=False, indent=2))
display(
    corrected_val_df.sort_values(
        ["accepted_correction", "char_distance_lower", "confidence"],
        ascending=[False, False, True],
    ).head(30)
)


,stage,split,cer,wer,ser,exact_match_case_sensitive,exact_match_case_insensitive,char_accuracy,avg_confidence,line_fps_batch_throughput,avg_line_latency_ms_batch,num_eval_samples,tag,decoder,beam_width,accepted_correction_ratio,avg_token_corrections
0,baseline_raw,grouped_val,0.019918,0.053734,0.274376,0.725624,0.741497,0.980082,0.987860,9.509544,105.157512,1323,baseline_raw_grouped_val,greedy,10.0,NaN,NaN
1,baseline_raw,camera_hard,0.562548,0.883919,0.886719,0.113281,0.113281,0.437452,0.960879,9.695937,103.135985,256,baseline_raw_camera_hard,greedy,10.0,NaN,NaN
2,baseline_plus_token_corrector,grouped_val,0.019588,0.052584,0.273621,0.726379,0.742252,0.980412,0.987860,NaN,NaN,1323,baseline_plus_token_corrector_grouped_val,NaN,NaN,0.011338,0.011338
3,baseline_plus_token_corrector,camera_hard,0.557602,0.872506,0.871094,0.128906,0.128906,0.442398,0.960879,NaN,NaN,256,baseline_plus_token_corrector_camera_hard,NaN,NaN,0.109375,0.109375


{
  "selected_stage": "baseline_plus_token_corrector",
  "backbone_weight_path": "/kaggle/input/models/trinhminhkhoak18hcm/ocr-train-chuan-checkpoint-weights/gguf/default/1/ocr-train-chuan_checkpoint.weights.h5",
  "token_corrector_model_path": "/kaggle/working/ocr_token_corrector_run/token_corrector.keras",
  "raw_baseline_width": 2304,
  "raw_baseline_decoder": "greedy",
  "raw_baseline_beam_width": 10,
  "grouped_val_metrics": {
    "cer": 0.019588275193957626,
    "wer": 0.052583752501496106,
    "ser": 0.273620559334845,
    "exact_match_case_sensitive": 0.7263794406651549,
    "exact_match_case_insensitive": 0.7422524565381708,
    "char_accuracy": 0.9804117248060423,
    "avg_confidence": 0.9878600191997923,
    "accepted_correction_ratio": 0.011337868480725623,
    "avg_token_corrections": 0.011337868480725623,
    "num_eval_samples": 1323,
    "tag": "baseline_plus_token_corrector_grouped_val"
  },
  "camera_hard_metrics": {
    "cer": 0.5576023033652153,
    "wer": 0.87250572

,split,filename,path,ground_truth,prediction,base_prediction,confidence,accepted_correction,num_token_corrections,token_events_json,lm_base,lm_candidate,char_distance_lower,word_distance_lower,exact_match
318,grouped_val,27_220_1.jpg,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,hộ dân không thường trú tại xã Xuân Hải.,ơ ha thông thông tra ại c huan hi,ơ ha thông thưông tra ại c huan hi,0.961951,1,1,"[{""position"": 6, ""base_token"": ""thưông"", ""cand...",0.677766,0.922037,16,9,0
727,grouped_val,79_2.jpg,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,Phước Thiện 4 Phước Sơn Ninh Phước Ninh Thuận,Phước Thiện 4 Phước Sơn Ninh Thuận,Phước Thiện 4 Phước Sơn Ninh Thưận,0.992299,1,1,"[{""position"": 12, ""base_token"": ""Thưận"", ""cand...",1.358345,1.634041,11,2,0
284,grouped_val,7_6_1.jpg,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,mưa quá lớn.,ca qua cun.,ca qhua cun.,0.975824,1,1,"[{""position"": 2, ""base_token"": ""qhua"", ""candid...",0.075791,0.558006,5,3,0
796,grouped_val,27_176_1.jpg,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,"ít kiềm chế được bản thân, còn người nhà họ vì...","1 kiêm chế được bản thơn, còn người nhà họ vì ...","1 kiêm chế được bản thơn, còn ngời nhà họ vì x...",0.975696,1,1,"[{""position"": 15, ""base_token"": ""ngời"", ""candi...",0.958376,1.162091,4,3,0
399,grouped_val,31_4_1.jpg,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,"hay không "" thỏa thuận ""?","huy không "" thủ thuận V?","huy không "" thủa thuận V?",0.988732,1,1,"[{""position"": 6, ""base_token"": ""thủa"", ""candid...",1.084779,1.298627,4,3,0
370,grouped_val,27_51_1.jpg,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,dày đặc.,dụng đc.,dng đc.,0.993929,1,1,"[{""position"": 0, ""base_token"": ""dng"", ""candida...",-0.180000,0.586019,4,2,0
705,grouped_val,195_10.jpg,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,Khu 4 Trạm Thản Phù Ninh Phú Thọ,Khuy 4 Tra Thản Phù Ninh Phú Thọ,Khuy 4 Tram Thản Phù Ninh Phú Thọ,0.979542,1,1,"[{""position"": 4, ""base_token"": ""Tram"", ""candid...",0.970750,1.165502,3,2,0
1075,grouped_val,0021_samples.png,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,"Tổ 1, Phường Yên Ninh, Thành phố Yên Bái, Yên Bái","Tổ 1, Phường Yên Ninh, Thành phố Yên Bai, Yồn Bái","Tổ 1, Phường Yiên Ninh, Thành phố Yên Bai, Yồn...",0.975367,1,1,"[{""position"": 7, ""base_token"": ""Yiên"", ""candid...",1.322854,1.506394,2,2,0
1100,grouped_val,1332_samples.png,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,"Thiệu Vận, Huyện Thiệu Hoá, Thanh Hoá","Thiệu Vân, Huyện Thuiệu Hoá, Thanh Hoá","Thiệu Vân, Huyện Thuiệu Hoá, Thanh Hoái",0.980556,1,1,"[{""position"": 14, ""base_token"": ""Hoái"", ""candi...",1.214513,1.414954,2,2,0
789,grouped_val,193_26.jpg,/kaggle/input/datasets/huyyuh/3500-w/vn_handwr...,Bình Phước Bình Sơn Quảng Ngãi,Bình Phước Bình Sơn Quảng Nai,Bình Phước Bình Sơn Quảng Ngai,0.988931,1,1,"[{""position"": 10, ""base_token"": ""Ngai"", ""candi...",1.672175,1.878542,2,1,0
